In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
import rasterio
from shapely.geometry import Point
import numpy as np
import pandas as pd
import os
from sklearn.preprocessing import RobustScaler
import pandas as pd
from shapely.geometry import box
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif

# Verification finales et correction des missing values

In [2]:
def impute_with_geo_zones(
    input_csv,
    num_cols=None,
    cat_cols=None,
    lat_col="latitude",
    lon_col="longitude",
    base_res=0.1,
    min_points=10,
    max_res=5.0,
    output_path=None,
):
    df = pd.read_csv(input_csv)
    num_cols = num_cols or []
    cat_cols = cat_cols or []

    # --- 1. Missing percentage ---
    missing_percent = df.isnull().mean() * 100
    print("Missing values (percent) per column :")
    print(missing_percent[missing_percent > 0])

    def impute_value(row, col, is_num, max_res):
        lat, lon = row[lat_col], row[lon_col]
        resolution = base_res

        while resolution <= max_res:
            lat_min, lat_max = lat - resolution, lat + resolution
            lon_min, lon_max = lon - resolution, lon + resolution

            # Vectorized filtering for the zone
            zone_df = df[
                (df[lat_col] >= lat_min)
                & (df[lat_col] <= lat_max)
                & (df[lon_col] >= lon_min)
                & (df[lon_col] <= lon_max)
            ]

            # Check the number of non-missing values in the target column
            valid_count = zone_df[col].count()

            if valid_count >= min_points:
                break
            resolution *= 1.5

        if valid_count == 0:  # Fallback to global
            print("Fallback to global , valid_count=0")
            return df[col].median() if is_num else df[col].mode(dropna=True).iat[0]

        if is_num:
            return zone_df[col].median()
        else:
            mode_val = zone_df[col].mode(dropna=True)
            return (
                mode_val.iat[0]
                if not mode_val.empty
                else df[col].mode(dropna=True).iat[0]
            )

    # --- 2. Process each column ---
    for col in df.columns:
        if col in [lat_col, lon_col] or missing_percent.get(col, 0) == 0:
            continue

        print(f"\n=== Imputing column: {col} ===")

        is_num = col in num_cols or (
            col not in cat_cols and pd.api.types.is_numeric_dtype(df[col])
        )

        # Select only the rows where the column is missing
        missing_rows = df[df[col].isnull()].copy()

        # Apply the imputation function to the missing rows
        imputed_values = missing_rows.apply(
            lambda row: impute_value(row, col, is_num, max_res), axis=1
        )

        # Assign the calculated values back to the original DataFrame
        df.loc[imputed_values.index, col] = imputed_values

        print(f"{col}: imputation done using geo-zones.")

    if output_path:
        df.to_csv(output_path, index=False)
        print(f"💾 Saved imputation to {output_path}")


In [3]:

def analyze_correlation_variance(csv_path, target_col="fire_count", corr_threshold=0.9):
    """
    Loads dataset, computes correlation matrix, finds correlated pairs,
    and computes variance of each feature.
    """
    df = pd.read_csv(csv_path)

    # Remove target + coordinates if needed
    numeric_df = df.drop(columns=[target_col, "latitude", "longitude"], errors="ignore")

    # --- 1. Compute correlation matrix
    corr_matrix = numeric_df.corr().abs()

    # --- 2. Extract pairs above threshold
    correlated_pairs = []
    for col1 in corr_matrix.columns:
        for col2 in corr_matrix.columns:
            if col1 < col2:  # avoid repetition
                if corr_matrix.loc[col1, col2] >= corr_threshold:
                    correlated_pairs.append((col1, col2, corr_matrix.loc[col1, col2]))

    # --- 3. Variance of each feature
    variances = numeric_df.var().sort_values(ascending=True)

    return {
        "correlated_pairs": correlated_pairs,
        "variances": variances,
        "corr_matrix": corr_matrix,
    }


def reduce_features(
    csv_path,
    output_path="reduced_dataset.csv",
    target_col="fire",
    var_threshold=0.01,
    corr_threshold=0.9,
    importance_method="RF",  # "MI" or "RF"
    top_k=20,  # number of best features to keep
):
    df = pd.read_csv(csv_path)

    X = df.drop(columns=[target_col], errors="ignore")
    y = df[target_col]

    # Remove non-informative ID columns
    X = X.drop(columns=["latitude", "longitude"], errors="ignore")

    # --- 1. Remove low variance features
    variances = X.var()
    keep_var = variances[variances > var_threshold].index
    X = X[keep_var]

    # --- 2. Remove highly correlated features smartly
    corr_matrix = X.corr().abs()

    # correlation of each feature with the target
    target_corr = df[X.columns].corrwith(y).abs()

    to_drop = set()
    cols = list(corr_matrix.columns)

    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            col1, col2 = cols[i], cols[j]
            if corr_matrix.loc[col1, col2] > corr_threshold:
                # drop the one LESS correlated with the label
                if target_corr[col1] >= target_corr[col2]:
                    to_drop.add(col2)
                else:
                    to_drop.add(col1)

    X = X.drop(columns=list(to_drop), errors="ignore")

    # --- 3. Feature importance (MI or RandomForest)
    if importance_method == "MI":
        scores = mutual_info_classif(X, y)
    else:
        rf = RandomForestClassifier(n_estimators=200, random_state=42)
        rf.fit(X, y)
        scores = rf.feature_importances_

    importance_df = pd.DataFrame({"feature": X.columns, "score": scores}).sort_values(
        by="score", ascending=False
    )

    # Keep top_k best features
    selected_features = importance_df.head(top_k)["feature"].tolist()

    # --- Output reduced dataset
    reduced_df = df[selected_features + [target_col]]
    reduced_df.to_csv(output_path, index=False)

    return {
        "selected_features": selected_features,
        "importance_table": importance_df,
        "output_path": output_path,
    }


In [4]:
elevation_path = "Cleaned_dataset/grid/elevation_grid.csv"
impute_with_geo_zones(
    input_csv=elevation_path,
    output_path=elevation_path,
)
climat_path1 = "Cleaned_dataset/grid/grid_tprec.csv"
impute_with_geo_zones(
    input_csv=climat_path1,
    output_path=climat_path1,
)   
climat_path2 = "Cleaned_dataset/grid/grid_tmax.csv"
impute_with_geo_zones(
    input_csv=climat_path2,
    output_path=climat_path2,
)
climat_path3 = "Cleaned_dataset/grid/grid_tmin.csv"
impute_with_geo_zones(
    input_csv=climat_path3,
    output_path=climat_path3,
)
soil_path = "Cleaned_dataset/grid/soil_grid.csv"


fire_path = "Cleaned_dataset/grid/fire_grid.csv"
impute_with_geo_zones(
    input_csv=fire_path,
    output_path=fire_path,
)
landcover_path = "Cleaned_dataset/grid/landcover_grid.csv"
impute_with_geo_zones(
    input_csv=landcover_path,
    output_path=landcover_path,
    cat_cols=['GRIDCODE']
)


Missing values (percent) per column :
Series([], dtype: float64)
💾 Saved imputation to Cleaned_dataset/grid/elevation_grid.csv
Missing values (percent) per column :
winter_prec    0.674777
spring_prec    0.674777
summer_prec    0.674777
autumn_prec    0.674777
dtype: float64

=== Imputing column: winter_prec ===
winter_prec: imputation done using geo-zones.

=== Imputing column: spring_prec ===
spring_prec: imputation done using geo-zones.

=== Imputing column: summer_prec ===
summer_prec: imputation done using geo-zones.

=== Imputing column: autumn_prec ===
autumn_prec: imputation done using geo-zones.
💾 Saved imputation to Cleaned_dataset/grid/grid_tprec.csv
Missing values (percent) per column :
winter_tmax    0.674777
spring_tmax    0.674777
summer_tmax    0.674777
autumn_tmax    0.674777
dtype: float64

=== Imputing column: winter_tmax ===
winter_tmax: imputation done using geo-zones.

=== Imputing column: spring_tmax ===
spring_tmax: imputation done using geo-zones.

=== Imputing

In [5]:
elev_df = pd.read_csv(elevation_path)
print(elev_df.shape)


climate_df1 = pd.read_csv(climat_path1)

climate_df1 = climate_df1.drop(columns=['year'], errors='ignore')
print(climate_df1.shape)

climate_df2 = pd.read_csv(climat_path2)

climate_df2 = climate_df2.drop(columns=['year'], errors='ignore')
print(climate_df2.shape)

climate_df3 = pd.read_csv(climat_path3)

climate_df3 = climate_df3.drop(columns=['year'], errors='ignore')
print(climate_df3.shape)

soil_df = pd.read_csv(soil_path)
print(soil_df.shape)

fire_df = pd.read_csv(fire_path)
fire_df = fire_df.drop(columns=['year'], errors='ignore')
print(fire_df.shape)

landcover_df = pd.read_csv(landcover_path)
print(landcover_df.shape)

(82694, 3)
(82694, 6)
(82694, 6)
(82694, 6)
(211489, 24)
(82694, 3)
(82694, 3)


In [6]:
merged_df = soil_df.copy()

merged_df = merged_df.merge(
    elev_df,
    on=['longitude', 'latitude'],
    how='left'
)



merged_df = merged_df.merge(
    climate_df1,
    on=['longitude', 'latitude'],
    how='left'
)

merged_df = merged_df.merge(
    climate_df2,
    on=['longitude', 'latitude'],
    how='left'
)

merged_df = merged_df.merge(
    climate_df3,
    on=['longitude', 'latitude'],
    how='left'
)


merged_df = merged_df.merge(
    landcover_df,
    on=['longitude', 'latitude'],
    how='left'
)


merged_df = merged_df.merge(
    fire_df,
    on=['longitude', 'latitude'],
    how='left'
)

print(merged_df.columns)

merged_df.to_csv("Cleaned_dataset/grid/MERGED_FINAL_without_norm.csv", index=False)

famd = merged_df.copy()
scaler = RobustScaler()
famd.to_csv("Cleaned_dataset/grid/MERGED_RAW.csv", index=False)

ignore_cols = [
    'GRIDCODE','TEXTURE_SOTER',
    'longitude', 'latitude'
]

num_cols = [c for c in famd.columns if c not in ignore_cols]


famd[num_cols] = scaler.fit_transform(famd[num_cols])
famd.to_csv("Cleaned_dataset/unsupervised/data_cl.csv", index=False)


Index(['longitude', 'latitude', 'HWSD2_SMU_ID', 'COARSE', 'SAND', 'SILT',
       'CLAY', 'TEXTURE_SOTER', 'BULK', 'REF_BULK', 'ORG_CARBON', 'PH_WATER',
       'TOTAL_N', 'CN_RATIO', 'CEC_SOIL', 'CEC_CLAY', 'CEC_EFF', 'TEB', 'BSAT',
       'ALUM_SAT', 'ESP', 'TCARBON_EQ', 'GYPSUM', 'ELEC_COND', 'elevation',
       'winter_prec', 'spring_prec', 'summer_prec', 'autumn_prec',
       'winter_tmax', 'spring_tmax', 'summer_tmax', 'autumn_tmax',
       'winter_tmin', 'spring_tmin', 'summer_tmin', 'autumn_tmin', 'GRIDCODE',
       'fire_count'],
      dtype='object')


In [7]:
merged_df.to_csv("Cleaned_dataset/grid/MERGED_FINAL.csv", index=False)

def one_hot_encode(csv_path, categorical_cols, label_col, output_path):
    """
    Reads a CSV and applies one-hot encoding.
    Ensures the new encoded features appear BEFORE the label column.
    """
    import pandas as pd

    df = pd.read_csv(csv_path)

    # Keep label aside
    label_series = df[label_col]

    # Apply OHE only on selected columns
    df_no_label = df.drop(columns=[label_col])
    df_encoded = pd.get_dummies(df_no_label, columns=categorical_cols, drop_first=False)

    # Reassemble with label at the end
    df_encoded[label_col] = label_series

    df_encoded.to_csv(output_path, index=False)
    return df_encoded

merged_df = one_hot_encode("Cleaned_dataset/grid/MERGED_FINAL.csv", ['GRIDCODE', 'TEXTURE_SOTER'], 'fire_count' ,"Cleaned_dataset/grid/MERGED_FINAL.csv")

print("Merged dataset shape:", merged_df.shape)

Merged dataset shape: (211489, 61)


In [8]:
print(merged_df.columns)

Index(['longitude', 'latitude', 'HWSD2_SMU_ID', 'COARSE', 'SAND', 'SILT',
       'CLAY', 'BULK', 'REF_BULK', 'ORG_CARBON', 'PH_WATER', 'TOTAL_N',
       'CN_RATIO', 'CEC_SOIL', 'CEC_CLAY', 'CEC_EFF', 'TEB', 'BSAT',
       'ALUM_SAT', 'ESP', 'TCARBON_EQ', 'GYPSUM', 'ELEC_COND', 'elevation',
       'winter_prec', 'spring_prec', 'summer_prec', 'autumn_prec',
       'winter_tmax', 'spring_tmax', 'summer_tmax', 'autumn_tmax',
       'winter_tmin', 'spring_tmin', 'summer_tmin', 'autumn_tmin',
       'GRIDCODE_14.0', 'GRIDCODE_16.0', 'GRIDCODE_20.0', 'GRIDCODE_30.0',
       'GRIDCODE_41.0', 'GRIDCODE_50.0', 'GRIDCODE_70.0', 'GRIDCODE_100.0',
       'GRIDCODE_110.0', 'GRIDCODE_120.0', 'GRIDCODE_130.0', 'GRIDCODE_134.0',
       'GRIDCODE_150.0', 'GRIDCODE_151.0', 'GRIDCODE_170.0', 'GRIDCODE_190.0',
       'GRIDCODE_200.0', 'GRIDCODE_201.0', 'GRIDCODE_202.0', 'GRIDCODE_203.0',
       'GRIDCODE_210.0', 'TEXTURE_SOTER_C', 'TEXTURE_SOTER_F',
       'TEXTURE_SOTER_M', 'fire_count'],
      dtype='obj

In [9]:
num_duplicates = merged_df.duplicated().sum()

print(f'The number of duplicated rows is: {num_duplicates}')

merged_df = merged_df.drop_duplicates()

num_duplicates = merged_df.duplicated().sum()

print(f'The number of duplicated rows is: {num_duplicates}')


The number of duplicated rows is: 0
The number of duplicated rows is: 0


In [10]:
unsuperved_df = merged_df.copy()
scaler = RobustScaler()

unsuperved_df.to_csv("Cleaned_dataset/grid/MERGED_RAW.csv", index=False)

ignore_cols = [
    'GRIDCODE_14.0','GRIDCODE_16.0','GRIDCODE_20.0','GRIDCODE_30.0',
    'GRIDCODE_41.0','GRIDCODE_50.0','GRIDCODE_70.0','GRIDCODE_100.0',
    'GRIDCODE_110.0','GRIDCODE_120.0','GRIDCODE_130.0','GRIDCODE_134.0',
    'GRIDCODE_150.0','GRIDCODE_151.0','GRIDCODE_170.0','GRIDCODE_190.0',
    'GRIDCODE_200.0','GRIDCODE_201.0','GRIDCODE_202.0','GRIDCODE_203.0',
    'GRIDCODE_210.0','TEXTURE_SOTER_C','TEXTURE_SOTER_F','TEXTURE_SOTER_M',
    'longitude', 'latitude'
]

num_cols = [c for c in unsuperved_df.columns if c not in ignore_cols]


unsuperved_df[num_cols] = scaler.fit_transform(unsuperved_df[num_cols])
unsuperved_df.to_csv("Cleaned_dataset/grid/MERGED_UNSUPERVED_long_lat.csv", index=False)

In [11]:
print(unsuperved_df.head)

<bound method NDFrame.head of         longitude   latitude  HWSD2_SMU_ID    COARSE      SAND      SILT  \
0       -1.653868  34.000231      0.000000 -0.615385 -0.466667  1.142857   
1       -1.653868  34.000231      0.000000  0.538462  1.266667 -1.428571   
2       -1.653868  34.000231      0.000000  0.076923  0.666667 -0.714286   
3       -1.653868  34.000231      0.000000  0.384615  0.000000 -0.285714   
4       -1.633868  34.000231      0.000000 -0.615385 -0.466667  1.142857   
...           ...        ...           ...       ...       ...       ...   
211484   9.846132  37.320231     40.513477 -0.153846  2.666667 -3.285714   
211485   9.846132  37.320231     40.513477  0.538462 -0.466667  0.571429   
211486   9.866132  37.320231      7.053908 -0.153846 -0.466667  0.571429   
211487   9.746132  37.340231     40.513477 -0.153846  2.666667 -3.285714   
211488   9.746132  37.340231     40.513477  0.538462 -0.466667  0.571429   

            CLAY      BULK  REF_BULK  ORG_CARBON  ...  GR

In [12]:
print(unsuperved_df.columns)
print(unsuperved_df.shape)

Index(['longitude', 'latitude', 'HWSD2_SMU_ID', 'COARSE', 'SAND', 'SILT',
       'CLAY', 'BULK', 'REF_BULK', 'ORG_CARBON', 'PH_WATER', 'TOTAL_N',
       'CN_RATIO', 'CEC_SOIL', 'CEC_CLAY', 'CEC_EFF', 'TEB', 'BSAT',
       'ALUM_SAT', 'ESP', 'TCARBON_EQ', 'GYPSUM', 'ELEC_COND', 'elevation',
       'winter_prec', 'spring_prec', 'summer_prec', 'autumn_prec',
       'winter_tmax', 'spring_tmax', 'summer_tmax', 'autumn_tmax',
       'winter_tmin', 'spring_tmin', 'summer_tmin', 'autumn_tmin',
       'GRIDCODE_14.0', 'GRIDCODE_16.0', 'GRIDCODE_20.0', 'GRIDCODE_30.0',
       'GRIDCODE_41.0', 'GRIDCODE_50.0', 'GRIDCODE_70.0', 'GRIDCODE_100.0',
       'GRIDCODE_110.0', 'GRIDCODE_120.0', 'GRIDCODE_130.0', 'GRIDCODE_134.0',
       'GRIDCODE_150.0', 'GRIDCODE_151.0', 'GRIDCODE_170.0', 'GRIDCODE_190.0',
       'GRIDCODE_200.0', 'GRIDCODE_201.0', 'GRIDCODE_202.0', 'GRIDCODE_203.0',
       'GRIDCODE_210.0', 'TEXTURE_SOTER_C', 'TEXTURE_SOTER_F',
       'TEXTURE_SOTER_M', 'fire_count'],
      dtype='obj

In [13]:
scaler = RobustScaler()

ignore_cols = [
    'GRIDCODE_14.0','GRIDCODE_16.0','GRIDCODE_20.0','GRIDCODE_30.0',
    'GRIDCODE_41.0','GRIDCODE_50.0','GRIDCODE_70.0','GRIDCODE_100.0',
    'GRIDCODE_110.0','GRIDCODE_120.0','GRIDCODE_130.0','GRIDCODE_134.0',
    'GRIDCODE_150.0','GRIDCODE_151.0','GRIDCODE_170.0','GRIDCODE_190.0',
    'GRIDCODE_200.0','GRIDCODE_201.0','GRIDCODE_202.0','GRIDCODE_203.0',
    'GRIDCODE_210.0','TEXTURE_SOTER_C','TEXTURE_SOTER_F','TEXTURE_SOTER_M'
]

num_cols = [c for c in merged_df.columns if c not in ignore_cols]


merged_df[num_cols] = scaler.fit_transform(merged_df[num_cols])


In [14]:
print(merged_df.columns)
print(merged_df.shape)

Index(['longitude', 'latitude', 'HWSD2_SMU_ID', 'COARSE', 'SAND', 'SILT',
       'CLAY', 'BULK', 'REF_BULK', 'ORG_CARBON', 'PH_WATER', 'TOTAL_N',
       'CN_RATIO', 'CEC_SOIL', 'CEC_CLAY', 'CEC_EFF', 'TEB', 'BSAT',
       'ALUM_SAT', 'ESP', 'TCARBON_EQ', 'GYPSUM', 'ELEC_COND', 'elevation',
       'winter_prec', 'spring_prec', 'summer_prec', 'autumn_prec',
       'winter_tmax', 'spring_tmax', 'summer_tmax', 'autumn_tmax',
       'winter_tmin', 'spring_tmin', 'summer_tmin', 'autumn_tmin',
       'GRIDCODE_14.0', 'GRIDCODE_16.0', 'GRIDCODE_20.0', 'GRIDCODE_30.0',
       'GRIDCODE_41.0', 'GRIDCODE_50.0', 'GRIDCODE_70.0', 'GRIDCODE_100.0',
       'GRIDCODE_110.0', 'GRIDCODE_120.0', 'GRIDCODE_130.0', 'GRIDCODE_134.0',
       'GRIDCODE_150.0', 'GRIDCODE_151.0', 'GRIDCODE_170.0', 'GRIDCODE_190.0',
       'GRIDCODE_200.0', 'GRIDCODE_201.0', 'GRIDCODE_202.0', 'GRIDCODE_203.0',
       'GRIDCODE_210.0', 'TEXTURE_SOTER_C', 'TEXTURE_SOTER_F',
       'TEXTURE_SOTER_M', 'fire_count'],
      dtype='obj

In [15]:
num_duplicates = merged_df.duplicated().sum()

print(f'The number of duplicated rows is: {num_duplicates}')

print(merged_df.columns)

The number of duplicated rows is: 0
Index(['longitude', 'latitude', 'HWSD2_SMU_ID', 'COARSE', 'SAND', 'SILT',
       'CLAY', 'BULK', 'REF_BULK', 'ORG_CARBON', 'PH_WATER', 'TOTAL_N',
       'CN_RATIO', 'CEC_SOIL', 'CEC_CLAY', 'CEC_EFF', 'TEB', 'BSAT',
       'ALUM_SAT', 'ESP', 'TCARBON_EQ', 'GYPSUM', 'ELEC_COND', 'elevation',
       'winter_prec', 'spring_prec', 'summer_prec', 'autumn_prec',
       'winter_tmax', 'spring_tmax', 'summer_tmax', 'autumn_tmax',
       'winter_tmin', 'spring_tmin', 'summer_tmin', 'autumn_tmin',
       'GRIDCODE_14.0', 'GRIDCODE_16.0', 'GRIDCODE_20.0', 'GRIDCODE_30.0',
       'GRIDCODE_41.0', 'GRIDCODE_50.0', 'GRIDCODE_70.0', 'GRIDCODE_100.0',
       'GRIDCODE_110.0', 'GRIDCODE_120.0', 'GRIDCODE_130.0', 'GRIDCODE_134.0',
       'GRIDCODE_150.0', 'GRIDCODE_151.0', 'GRIDCODE_170.0', 'GRIDCODE_190.0',
       'GRIDCODE_200.0', 'GRIDCODE_201.0', 'GRIDCODE_202.0', 'GRIDCODE_203.0',
       'GRIDCODE_210.0', 'TEXTURE_SOTER_C', 'TEXTURE_SOTER_F',
       'TEXTURE_SOTER

In [16]:
merged_df

,longitude,latitude,HWSD2_SMU_ID,COARSE,SAND,SILT,CLAY,BULK,REF_BULK,ORG_CARBON,...,GRIDCODE_190.0,GRIDCODE_200.0,GRIDCODE_201.0,GRIDCODE_202.0,GRIDCODE_203.0,GRIDCODE_210.0,TEXTURE_SOTER_C,TEXTURE_SOTER_F,TEXTURE_SOTER_M,fire_count
0,-1.147260,-0.931507,0.000000,-0.615385,-0.466667,1.142857,0.000000,0.555555,-0.058823,-0.037691,...,False,False,True,False,False,False,False,False,True,1.0
1,-1.147260,-0.931507,0.000000,0.538462,1.266667,-1.428571,-0.666667,-0.777778,-0.941177,-0.018846,...,False,False,True,False,False,False,True,False,False,1.0
2,-1.147260,-0.931507,0.000000,0.076923,0.666667,-0.714286,-0.333333,-0.111112,-0.411765,-0.130742,...,False,False,True,False,False,False,False,False,True,1.0
3,-1.147260,-0.931507,0.000000,0.384615,0.000000,-0.285714,0.250000,-0.222223,0.235294,-0.214370,...,False,False,True,False,False,False,False,False,True,1.0
4,-1.143836,-0.931507,0.000000,-0.615385,-0.466667,1.142857,0.000000,0.555555,-0.058823,-0.037691,...,False,False,True,False,False,False,False,False,True,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
211484,0.821918,1.342466,40.513477,-0.153846,2.666667,-3.285714,-1.333333,0.000000,-3.058822,-0.116608,...,False,False,False,False,False,False,True,False,False,1.0
211485,0.821918,1.342466,40.513477,0.538462,-0.466667,0.571429,0.333333,-0.222223,0.294117,0.531213,...,False,False,False,False,False,False,False,False,True,1.0
211486,0.825342,1.342466,7.053908,-0.153846,-0.466667,0.571429,0.333333,-0.222223,-0.058823,0.435218,...,False,False,False,False,False,True,False,False,True,0.0
211487,0.804795,1.356164,40.513477,-0.153846,2.666667,-3.285714,-1.333333,0.000000,-3.058822,-0.116608,...,False,False,False,False,False,False,True,False,False,0.0


In [17]:
merged_df.to_csv("Cleaned_dataset/grid/MERGED_FINAL.csv", index=False)
merged_df.to_csv("Cleaned_dataset/grid/MERGED_UNSUPERVED.csv", index=False)

In [18]:
reduced = reduce_features(
    "Cleaned_dataset/grid/MERGED_FINAL.csv",
    output_path="Cleaned_dataset/grid/MERGED_FINAL.csv",
    target_col="fire_count",
    var_threshold=0.01,
    corr_threshold=0.95,
    importance_method="RF",
    top_k=40
)

print("Selected features:", reduced["selected_features"])

Selected features: ['spring_prec', 'autumn_prec', 'winter_prec', 'summer_prec', 'autumn_tmax', 'elevation', 'summer_tmax', 'autumn_tmin', 'spring_tmax', 'winter_tmin', 'spring_tmin', 'summer_tmin', 'HWSD2_SMU_ID', 'TOTAL_N', 'winter_tmax', 'ORG_CARBON', 'TCARBON_EQ', 'CEC_EFF', 'SAND', 'PH_WATER', 'CN_RATIO', 'REF_BULK', 'SILT', 'GRIDCODE_30.0', 'ESP', 'GRIDCODE_20.0', 'CEC_SOIL', 'CEC_CLAY', 'GRIDCODE_150.0', 'BULK', 'COARSE', 'GYPSUM', 'GRIDCODE_14.0', 'GRIDCODE_130.0', 'BSAT', 'GRIDCODE_201.0', 'GRIDCODE_151.0', 'ELEC_COND', 'GRIDCODE_110.0', 'GRIDCODE_200.0']


In [19]:
merged_df = pd.read_csv("Cleaned_dataset/grid/MERGED_FINAL.csv")

In [20]:
merged_df = pd.read_csv("Cleaned_dataset/grid/MERGED_FINAL.csv")
merged_df = merged_df.drop(columns=["HWSD2_SMU_ID"], errors='ignore')
merged_df

,spring_prec,autumn_prec,winter_prec,summer_prec,autumn_tmax,elevation,summer_tmax,autumn_tmin,spring_tmax,winter_tmin,...,GYPSUM,GRIDCODE_14.0,GRIDCODE_130.0,BSAT,GRIDCODE_201.0,GRIDCODE_151.0,ELEC_COND,GRIDCODE_110.0,GRIDCODE_200.0,fire_count
0,2.961474,1.542345,0.564423,0.628049,-0.923077,0.622003,-0.50,-1.25000,-0.666667,-1.333333,...,0.052632,False,False,-0.090909,True,False,1.0,False,False,1.0
1,2.961474,1.542345,0.564423,0.628049,-0.923077,0.622003,-0.50,-1.25000,-0.666667,-1.333333,...,-0.105263,False,False,-1.590909,True,False,0.0,False,False,1.0
2,2.961474,1.542345,0.564423,0.628049,-0.923077,0.622003,-0.50,-1.25000,-0.666667,-1.333333,...,1.842105,False,False,0.045455,True,False,0.0,False,False,1.0
3,2.961474,1.542345,0.564423,0.628049,-0.923077,0.622003,-0.50,-1.25000,-0.666667,-1.333333,...,3.315790,False,False,0.045455,True,False,13.0,False,False,1.0
4,-0.251256,-0.358306,0.002885,1.256098,-0.461538,0.631876,0.00,-0.75000,0.333333,-0.800000,...,0.052632,False,False,-0.090909,True,False,1.0,False,False,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
211484,0.400335,1.159609,2.499038,-0.219512,0.615385,-0.895628,-0.75,0.75000,-0.333333,0.866667,...,-0.210526,False,False,-1.045455,False,False,-1.0,False,False,1.0
211485,0.400335,1.159609,2.499038,-0.219512,0.615385,-0.895628,-0.75,0.75000,-0.333333,0.866667,...,-0.105263,False,False,0.045455,False,False,-1.0,False,False,1.0
211486,0.400335,1.159609,2.499038,-0.219512,0.615385,-0.903385,-0.75,0.75000,-0.333333,0.866667,...,-0.210526,False,False,-0.727273,False,False,-1.0,False,False,0.0
211487,1.361809,2.052117,2.499038,-0.189024,0.346154,-0.872355,-1.00,0.71875,-0.333333,0.866667,...,-0.210526,True,False,-1.045455,False,False,-1.0,False,False,0.0


In [21]:
merged_df.to_csv("Cleaned_dataset/grid/MERGED_FINAL.csv")

In [22]:
num_duplicates = merged_df.duplicated().sum()

print(f'The number of duplicated rows is: {num_duplicates}')

print(merged_df.columns)
print(merged_df.shape)

merged_df.to_csv("Cleaned_dataset/grid/MERGED_FINAL.csv")

The number of duplicated rows is: 8829
Index(['spring_prec', 'autumn_prec', 'winter_prec', 'summer_prec',
       'autumn_tmax', 'elevation', 'summer_tmax', 'autumn_tmin', 'spring_tmax',
       'winter_tmin', 'spring_tmin', 'summer_tmin', 'TOTAL_N', 'winter_tmax',
       'ORG_CARBON', 'TCARBON_EQ', 'CEC_EFF', 'SAND', 'PH_WATER', 'CN_RATIO',
       'REF_BULK', 'SILT', 'GRIDCODE_30.0', 'ESP', 'GRIDCODE_20.0', 'CEC_SOIL',
       'CEC_CLAY', 'GRIDCODE_150.0', 'BULK', 'COARSE', 'GYPSUM',
       'GRIDCODE_14.0', 'GRIDCODE_130.0', 'BSAT', 'GRIDCODE_201.0',
       'GRIDCODE_151.0', 'ELEC_COND', 'GRIDCODE_110.0', 'GRIDCODE_200.0',
       'fire_count'],
      dtype='object')
(211489, 40)


In [23]:
print("\n================ FIRE_COUNT DISTRIBUTION ================")
print(merged_df['fire_count'].value_counts().sort_index())
print("==========================================================")


================ FIRE_COUNT DISTRIBUTION ================
fire_count
0.0    176425
1.0     35064
Name: count, dtype: int64


# VISUALIZATION

In [24]:
merged_df

,spring_prec,autumn_prec,winter_prec,summer_prec,autumn_tmax,elevation,summer_tmax,autumn_tmin,spring_tmax,winter_tmin,...,GYPSUM,GRIDCODE_14.0,GRIDCODE_130.0,BSAT,GRIDCODE_201.0,GRIDCODE_151.0,ELEC_COND,GRIDCODE_110.0,GRIDCODE_200.0,fire_count
0,2.961474,1.542345,0.564423,0.628049,-0.923077,0.622003,-0.50,-1.25000,-0.666667,-1.333333,...,0.052632,False,False,-0.090909,True,False,1.0,False,False,1.0
1,2.961474,1.542345,0.564423,0.628049,-0.923077,0.622003,-0.50,-1.25000,-0.666667,-1.333333,...,-0.105263,False,False,-1.590909,True,False,0.0,False,False,1.0
2,2.961474,1.542345,0.564423,0.628049,-0.923077,0.622003,-0.50,-1.25000,-0.666667,-1.333333,...,1.842105,False,False,0.045455,True,False,0.0,False,False,1.0
3,2.961474,1.542345,0.564423,0.628049,-0.923077,0.622003,-0.50,-1.25000,-0.666667,-1.333333,...,3.315790,False,False,0.045455,True,False,13.0,False,False,1.0
4,-0.251256,-0.358306,0.002885,1.256098,-0.461538,0.631876,0.00,-0.75000,0.333333,-0.800000,...,0.052632,False,False,-0.090909,True,False,1.0,False,False,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
211484,0.400335,1.159609,2.499038,-0.219512,0.615385,-0.895628,-0.75,0.75000,-0.333333,0.866667,...,-0.210526,False,False,-1.045455,False,False,-1.0,False,False,1.0
211485,0.400335,1.159609,2.499038,-0.219512,0.615385,-0.895628,-0.75,0.75000,-0.333333,0.866667,...,-0.105263,False,False,0.045455,False,False,-1.0,False,False,1.0
211486,0.400335,1.159609,2.499038,-0.219512,0.615385,-0.903385,-0.75,0.75000,-0.333333,0.866667,...,-0.210526,False,False,-0.727273,False,False,-1.0,False,False,0.0
211487,1.361809,2.052117,2.499038,-0.189024,0.346154,-0.872355,-1.00,0.71875,-0.333333,0.866667,...,-0.210526,True,False,-1.045455,False,False,-1.0,False,False,0.0
